In [ ]:
#!/usr/bin/env python3
# pip install seqeval
import os
import time
import asyncio
import aiohttp
import pandas as pd
from google.colab import drive
from seqeval.metrics import classification_report, f1_score, precision_score, recall_score
import nest_asyncio

nest_asyncio.apply()

API_KEY         = ""
OPENROUTER_URL  = "https://openrouter.ai/api/v1/chat/completions"
CSV_PATH        = "stratified_sample_200.csv"
MODEL           = "google/gemini-3.6-flash"
OUTPUT_DIR      = "/content"
NUM_SAMPLES     = 200
CONCURRENCY     = 10
BATCH_SIZE      = 10
BATCH_DELAY     = 2
SITE_URL        = "https://colab.research.google.com"
SITE_NAME       = "Bangla NER Benchmark"
MAX_RETRIES     = 2
BASE_BACKOFF    = 2.0
REQUEST_TIMEOUT = 90

GT_LABEL_MAP = {
    "B-Health":   "B-Health_Condition",
    "I-Health":   "I-Health_Condition",
    "B-Medical":  "B-Medical_Procedure",
    "I-Medical":  "I-Medical_Procedure",
}

VALID_LABELS = [
    "O",
    "B-Symptom",            "I-Symptom",
    "B-Health_Condition",   "I-Health_Condition",
    "B-Medicine",           "I-Medicine",
    "B-Specialist",         "I-Specialist",
    "B-Age",                "I-Age",
    "B-Dosage",             "I-Dosage",
    "B-Medical_Procedure",  "I-Medical_Procedure",
]
VALID_SET   = set(VALID_LABELS)
VALID_LOWER = {v.lower(): v for v in VALID_LABELS}

# ── 8-shot examples (token → label pairs) ──────────────────────────────────
FEW_SHOT_EXAMPLES = [
    # 1. Symptom
    {
        "tokens": ["স্যার", ":", "আমার", "অনেক", "দিন", "ধরে", "সর্দিতে", "নাক", "বন্দ", "আছে", "।", "এখন", "মাথা", "বার", "আর", "ব্যথা", "।"],
        "labels": ["O", "O", "O", "O", "O", "O", "B-Symptom", "I-Symptom", "I-Symptom", "I-Symptom", "O", "O", "B-Symptom", "I-Symptom", "I-Symptom", "I-Symptom", "O"],
    },
    # 2. Age
    {
        "tokens": ["আমার", "বয়স", "২৪", "।", "আমার", "৮", "দিন", "ধরে", "হালকা", "খুশ", "খুশে", "কাশি", "।"],
        "labels": ["O", "B-Age", "I-Age", "O", "O", "O", "O", "O", "B-Symptom", "I-Symptom", "I-Symptom", "I-Symptom", "O"],
    },
    # 3. Medicine + Dosage
    {
        "tokens": ["Tab", ".", "Omidon", "10mg", "1", "+", "1", "+", "1", "before", "meal", "for", "5", "days"],
        "labels": ["B-Medicine", "I-Medicine", "I-Medicine", "I-Medicine", "B-Dosage", "I-Dosage", "I-Dosage", "I-Dosage", "I-Dosage", "I-Dosage", "I-Dosage", "I-Dosage", "I-Dosage", "I-Dosage"],
    },
    # 4. Medical Procedure + Health Condition
    {
        "tokens": ["আমার", "মায়ের", "ডায়াবেটিস", "টেষ্ট", "করি", "।", "মায়ের", "আগে", "থেকেই", "ডায়াবেটিস", "আছে", "।"],
        "labels": ["O", "O", "B-Medical_Procedure", "I-Medical_Procedure", "O", "O", "O", "O", "O", "B-Health_Condition", "O", "O"],
    },
    # 5. Specialist
    {
        "tokens": ["একজন", "মেডিসিন", "বিশেষজ্ঞ", "চিকিৎসক", "এর", "পরামর্শ", "গ্রহন", "করুন", "।"],
        "labels": ["O", "B-Specialist", "I-Specialist", "O", "O", "O", "O", "O", "O"],
    },
    # 6. Multiple Symptoms
    {
        "tokens": ["ওর", "গলায়", "ডোক", "গিল্লে", "ব্যথা", "করে", "।", "গলার", "ভিতরে", "টন্সিল", "একটু", "বড়", "হয়েছে", "।", "বুকে", "কফ", "আছে", "।"],
        "labels": ["O", "B-Symptom", "I-Symptom", "I-Symptom", "I-Symptom", "O", "O", "B-Symptom", "I-Symptom", "I-Symptom", "I-Symptom", "I-Symptom", "I-Symptom", "O", "B-Symptom", "I-Symptom", "O", "O"],
    },
    # 7. Medical Procedure
    {
        "tokens": ["একজন", "ডাক্তার", "দিয়ে", "ব্রেস্ট", "পরীক্ষা", "করিয়ে", "নিন", "।"],
        "labels": ["O", "O", "O", "B-Medical_Procedure", "I-Medical_Procedure", "O", "O", "O"],
    },
    # 8. No entities (O-only)
    {
        "tokens": ["ধন্যবাদ", "আপনাকে", "প্রশ্নের", "জন্য", "।", "ডাবের", "পানি", "স্যালাইন", "দুধ", "ডিম", "মুরগির", "সুপ", "খেতে", "পারেন", "।"],
        "labels": ["O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O", "O"],
    },
]

SYSTEM_PROMPT = """You are a Bangla Biomedical Named Entity Recognition (BioNER) model.

Label each input token using the IOB2 scheme.

Entity types and their tags:
- Symptom          → B-Symptom, I-Symptom
- Health Condition → B-Health_Condition, I-Health_Condition
- Medicine         → B-Medicine, I-Medicine
- Dosage           → B-Dosage, I-Dosage
- Medical Procedure→ B-Medical_Procedure, I-Medical_Procedure
- Specialist       → B-Specialist, I-Specialist
- Age              → B-Age, I-Age
- Outside          → O

Rules:
- Output exactly one tag per input token, one per line.
- Preserve token order strictly.
- Use B- for the first token of an entity span.
- Use I- for continuation tokens of the same entity.
- Output O for non-entity tokens.
- Return ONLY the labels, nothing else."""


def build_few_shot_block() -> str:
    """Build the 8-shot example block to prepend to every user message."""
    blocks = []
    for i, ex in enumerate(FEW_SHOT_EXAMPLES, 1):
        sentence   = " ".join(ex["tokens"])
        token_list = "\n".join(ex["tokens"])
        label_list = "\n".join(ex["labels"])
        blocks.append(
            f"### Example {i}\n"
            f"Sentence:\n{sentence}\n\n"
            f"Tokens:\n{token_list}\n\n"
            f"Labels:\n{label_list}"
        )
    return "\n\n".join(blocks)


FEW_SHOT_BLOCK = build_few_shot_block()   # built once at import time


def build_user_message(tokens: list[str]) -> str:
    sentence   = " ".join(tokens)
    token_list = "\n".join(tokens)

    return (
        f"{FEW_SHOT_BLOCK}\n\n"
        "### Now label the following\n"
        f"Sentence:\n{sentence}\n\n"
        "Assign one IOB2 label to each token below.\n"
        "Return ONLY the labels in order, one per line.\n\n"
        f"Tokens:\n{token_list}"
    )


def parse_ground_truth(label_str):
    parts  = str(label_str).split(" ")
    labels = []
    j      = 0
    while j < len(parts):
        token = parts[j]
        if token in GT_LABEL_MAP:
            labels.append(GT_LABEL_MAP[token])
            j += 2
        else:
            labels.append(token)
            j += 1
    return labels


def parse_response(raw_text, num_tokens):
    if raw_text is None:
        return ["O"] * num_tokens
    lines  = [l.strip() for l in raw_text.strip().split("\n") if l.strip()]
    parsed = []
    for line in lines:
        if line in VALID_SET:
            parsed.append(line)
        else:
            parsed.append(VALID_LOWER.get(line.lower(), "O"))
    if len(parsed) < num_tokens:
        parsed += ["O"] * (num_tokens - len(parsed))
    return parsed[:num_tokens]


def build_headers():
    return {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type":  "application/json",
        "HTTP-Referer":  SITE_URL,
        "X-Title":       SITE_NAME,
    }


async def call_model_async(session, model, user_message, headers, num_tokens):
    payload = {
      "model": model,
      "messages": [
          {"role": "system", "content": SYSTEM_PROMPT},
          {"role": "user",   "content": user_message},
      ],
      "max_completion_tokens": 1500,
      "reasoning_effort": "minimal",

      # 3. Explicit fallback to cap the hidden thinking budget via OpenRouter parameters

    }

    start = time.monotonic()
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            async with session.post(
                OPENROUTER_URL, headers=headers, json=payload,
                timeout=aiohttp.ClientTimeout(total=REQUEST_TIMEOUT)
            ) as resp:
                if resp.status == 429:
                    wait = float(resp.headers.get("Retry-After", BASE_BACKOFF * attempt))
                    await asyncio.sleep(wait)
                    continue
                if resp.status >= 500:
                    await asyncio.sleep(BASE_BACKOFF * attempt)
                    continue
                if resp.status != 200:
                    text = await resp.text()
                    return None, f"http_error:{resp.status}:{text[:200]}", time.monotonic() - start
                data    = await resp.json()
                content = data["choices"][0]["message"]["content"]

                # ── reasoning token visibility ──
                usage   = data.get("usage", {})
                details = usage.get("completion_tokens_details", {})
                reasoning_tokens = details.get("reasoning_tokens", 0)
                output_tokens    = usage.get("completion_tokens", "?")
                total_tokens     = usage.get("total_tokens", "?")
                print(f"  [tokens] reasoning={reasoning_tokens} output={output_tokens} total={total_tokens} | reasoning_on={reasoning_tokens > 0}")

                return content, None, time.monotonic() - start
        except Exception as e:
            if attempt == MAX_RETRIES:
                return None, f"request_error:{e}", time.monotonic() - start
            await asyncio.sleep(BASE_BACKOFF * attempt)
    return None, "max_retries_exceeded", time.monotonic() - start


def checkpoint_path(out_dir, model):
    return os.path.join(out_dir, f"checkpoint_{model.replace('/', '__')}.csv")

def load_checkpoint(out_dir, model):
    path = checkpoint_path(out_dir, model)
    return pd.read_csv(path) if os.path.exists(path) else None

def save_checkpoint(out_dir, model, rows):
    pd.DataFrame(rows).to_csv(checkpoint_path(out_dir, model), index=False)


async def run_model_async(model, df, headers, out_dir):
    existing  = load_checkpoint(out_dir, model)
    rows_done = existing.to_dict("records") if existing is not None else []
    start_idx = len(rows_done)

    if start_idx >= len(df):
        print("Already complete — loaded from checkpoint.")
        return rows_done

    sem     = asyncio.Semaphore(CONCURRENCY)
    indices = list(range(start_idx, len(df)))
    batches = [indices[i:i + BATCH_SIZE] for i in range(0, len(indices), BATCH_SIZE)]
    results = []

    async def process_row(session, row_idx):
        async with sem:
            row         = df.iloc[row_idx]
            tokens      = str(row["text"]).split()
            true_labels = parse_ground_truth(row["labels"])

            min_len     = min(len(tokens), len(true_labels))
            tokens      = tokens[:min_len]
            true_labels = true_labels[:min_len]

            user_msg = build_user_message(tokens)
            raw, err, latency = await call_model_async(session, model, user_msg, headers, min_len)
            pred_labels = parse_response(raw, min_len)

            min_len     = min(len(true_labels), len(pred_labels))
            true_labels = true_labels[:min_len]
            pred_labels = pred_labels[:min_len]

            print(f"[{row_idx}] tokens={len(tokens)} pred_lines={len(raw.splitlines()) if raw else 0} err={err}")

            return {
                "_idx":          row_idx,
                "text":          row["text"],
                "true_labels":   " ".join(true_labels),
                "pred_labels":   " ".join(pred_labels),
                "primary_class": row.get("primary_class", ""),
                "latency":       latency,
                "error":         err,
            }

    async with aiohttp.ClientSession() as session:
        for b_num, batch in enumerate(batches):
            print(f"\nBatch {b_num + 1}/{len(batches)} — rows {batch[0]}..{batch[-1]}")
            tasks         = [process_row(session, i) for i in batch]
            batch_results = await asyncio.gather(*tasks)
            results.extend(batch_results)

            all_rows = rows_done + sorted(results, key=lambda r: r["_idx"])
            save_checkpoint(out_dir, model, all_rows)

            if b_num < len(batches) - 1:
                print(f"Waiting {BATCH_DELAY}s…")
                await asyncio.sleep(BATCH_DELAY)

    results  = sorted(results, key=lambda r: r["_idx"])
    for r in results:
        del r["_idx"]
    all_rows = rows_done + results
    save_checkpoint(out_dir, model, all_rows)
    return all_rows


def run_model_on_dataset(model, df, headers, out_dir):
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(run_model_async(model, df, headers, out_dir))


def compute_metrics(rows):
    y_true, y_pred               = [], []
    all_true_flat, all_pred_flat = [], []
    latencies                    = []

    for r in rows:
        if pd.notna(r.get("true_labels")) and pd.notna(r.get("pred_labels")):
            t = str(r["true_labels"]).split()
            p = str(r["pred_labels"]).split()
            min_len = min(len(t), len(p))
            y_true.append(t[:min_len])
            y_pred.append(p[:min_len])
            all_true_flat.extend(t[:min_len])
            all_pred_flat.extend(p[:min_len])
        if r.get("latency") is not None:
            latencies.append(r["latency"])

    correct        = sum(t == p for t, p in zip(all_true_flat, all_pred_flat))
    token_accuracy = correct / len(all_true_flat) if all_true_flat else 0

    return {
        "token_accuracy": token_accuracy,
        "f1":             f1_score(y_true, y_pred, average="weighted"),
        "precision":      precision_score(y_true, y_pred, average="weighted"),
        "recall":         recall_score(y_true, y_pred, average="weighted"),
        "total":          len(rows),
        "avg_latency":    sum(latencies) / len(latencies) if latencies else None,
        "report":         classification_report(y_true, y_pred, digits=4),
    }


def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    df = pd.read_csv(CSV_PATH)
    if not {"text", "labels"}.issubset(df.columns):
        raise ValueError("CSV must contain 'text' and 'labels' columns")

    if NUM_SAMPLES is not None:
        df = df.head(NUM_SAMPLES).reset_index(drop=True)

    print(f"\nDataset: {len(df)} sentences")
    print(f"Prompting strategy: 8-shot")
    from collections import Counter
    all_labels = [lbl for row in df["labels"] for lbl in str(row).split()]
    print("Label distribution:")
    for lbl, cnt in sorted(Counter(all_labels).items()):
        print(f"  {lbl:<30} {cnt}")

    headers = build_headers()
    rows    = run_model_on_dataset(MODEL, df, headers, OUTPUT_DIR)
    metrics = compute_metrics(rows)
    model_slug = MODEL.replace("/", "_")
    size_tag   = str(NUM_SAMPLES) if NUM_SAMPLES is not None else "full"

    pd.DataFrame(rows).to_csv(
    os.path.join(OUTPUT_DIR, f"{model_slug}_{size_tag}_predictions_fewshot.csv"), index=False, encoding="utf-8-sig")

    results_df = pd.DataFrame([{
        "model":              MODEL,
        "prompting":          "8-shot",
        "token_accuracy":     metrics["token_accuracy"],
        "f1_weighted":        metrics["f1"],
        "precision_weighted": metrics["precision"],
        "recall_weighted":    metrics["recall"],
        "total_sentences":    metrics["total"],
        "avg_latency_s":      metrics["avg_latency"],
    }])
    results_df.to_csv(
    os.path.join(OUTPUT_DIR, f"{model_slug}_{size_tag}_benchmark_results_fewshot.csv"), index=False, encoding="utf-8-sig")

    print("\n── Overall Results ──")
    print(results_df.to_string(index=False))
    print("\n── seqeval Span-Level Report (per entity type) ──")
    print(metrics["report"])


if __name__ == "__main__":
    main()


Dataset: 196 sentences
Prompting strategy: 8-shot
Label distribution:
  B-Age                          46
  B-Dosage                       88
  B-Health                       65
  B-Medical                      49
  B-Medicine                     100
  B-Specialist                   57
  B-Symptom                      107
  Condition                      136
  I-Age                          58
  I-Dosage                       287
  I-Health                       71
  I-Medical                      71
  I-Medicine                     165
  I-Specialist                   67
  I-Symptom                      336
  O                              5100
  Procedure                      120

Batch 1/20 — rows 0..9
  [tokens] reasoning=0 output=65 total=1303 | reasoning_on=False
[2] tokens=19 pred_lines=19 err=None
  [tokens] reasoning=0 output=105 total=1373 | reasoning_on=False
[3] tokens=26 pred_lines=26 err=None
  [tokens] reasoning=0 output=31 total=1255 | reasoning_on=False
[5] tokens=12 